# Functional dependencies and arithmetic identities

A functional dependency is a mapping claim: equal inputs correspond to
equal outputs, subject to recorded exceptions and scope. The profiler
discovers them with a three-stage pipeline — cheap screening, bounded
lattice search, exact verification — and reports two forms: `value_mapping`
(lookup-style dependencies, exact or approximate) and `affine`
(arithmetic identities `y = a·x + b`). Both are `dependencies/functional`
statements; a high predictive score never upgrades a claim to this
category.

## Semantics

FDs are evaluated on tuple-complete rows: a row with a null on the
determinant or the determined side is excluded, and `n_used` records how
many rows the claim actually covers. Equality is canonical: value bytes,
not display strings, so `1.0` and `1` compare equal as i64/f64 bit
patterns dictate and strings compare by bytes. Both policies are stated in
the finding's specification, because both change what the claim means on
nullable, mixed-format data.

## Discovery: screen, lattice, verify

**Screen.** Every eligible column is hashed to 64-bit fingerprints over a
2,048-row stratified sample. Eligibility for a determinant: at least two
distinct values, thirty non-null rows, a usable role, a distinct ratio
under 0.98 (near-unique columns are keys, not lookups), and at most
100,000 distinct values. For every (determinant, target) pair the screen
tracks the top two target values per determinant group — enough to
estimate violations without keeping groups in memory. Per target, at most
16 determinants (32 on `deep`) survive, ordered categorical-first then by
distinct count.

**Lattice.** Determinant sets grow level-wise — singletons, then pairs
(then triples on `deep`, drawn from the 12 columns that produced the most
exact dependencies). Subset pruning does the combinatorial work: if a
subset of a candidate already determines the target, the candidate is
skipped; if an exact dependency is found, everything it implies is
closed off. One honesty rule lives here: a determinant that is itself
nearly unique (its groups cover ≥ 95% of rows) trivially determines every
target, so it is excluded — that is the key story, not a dozen spurious
dependencies.

**Verify.** Every emitted candidate is re-verified with canonical value
bytes over up to 200,000 stratified rows (500,000 on `deep`) — hashing is
only ever allowed to cost recall, never truth. A candidate that fails
becomes an approximate dependency if its error is small enough; a
candidate that passes is exact on the verified rows, and the finding says
which rows those were.

## Approximate dependencies

The profiler reports the g1 error — the share of rows that would have to
change for the dependency to hold — and keeps an approximate dependency
only when g1 ≤ 0.01. At most one approximate dependency per target
survives (the smallest error, preferring simpler determinants), and one
suppression keeps the ledger honest: an approximate dependency whose
determinants already contain an exact dependency's determinants is
dropped, because the exact rule explains the exceptions better than the
approximate rule restates the mapping. The finding carries the violation
count so a reader can go from claim to rows.

In [ ]:
import random

import polars as pl

import onceover

rng = random.Random(9)
n = 3_000
region_id = [rng.randrange(3) for _ in range(n)]
manager = [["Ann", "Bob", "Cid"][r] for r in region_id]
for i in (100, 900, 1700):          # exactly three violations
    manager[i] = "Dana"
df = pl.DataFrame({
    "region_id": region_id,
    "manager": manager,
    "noise": [rng.random() for _ in range(n)],
})

report = onceover.profile(df)
for f in report.findings:
    if f["category"] == "dependencies" and f.get("type") == "functional" and "manager" in f["subject"]["columns"]:
        e = f["evidence"][0]
        print(f"{f['claim']}")
        print(f"   violations={e['metrics']['violations']} n_used={e['metrics']['n_used']}")

## Arithmetic identities

The identity probe looks for the affine law `y = a·x + b` between numeric
columns. Candidates are pairs the dependence screen flagged (|Pearson| ≥
0.8 or ξ ≥ 0.9, at most 100 pairs on `standard`, 200 on `deep`); the fit
is ordinary least squares over a ≤ 20,000-row sample; and the claim is
checked **on every pair-complete row** — residuals within 1e-9 absolute
plus 1e-9 relative of the fitted line. An identity is reported when at
least 99.9% of rows match, with the coefficients, the tolerance, the
matching-row count, and any violating rows in the metrics. The statement
is `dependencies/functional` with form `affine`; the finding's
interpretation names the practical consequence — a derived column, or a
check column.

Pairwise affine is the deliberate scope: it is the shape that covers the
common derived columns (totals, conversions, scaled units). Multivariate
laws and nonlinear closed forms are future work
([](../appendix/directions.ipynb)).

In [ ]:
df = pl.DataFrame({
    "quantity": [rng.randrange(1, 11) for _ in range(n)],
})
df = df.with_columns((2 * pl.col("quantity")).alias("double_quantity"))
report = onceover.profile(df)
identity = next(f for f in report.findings if f["specification"].get("form") == "affine")
print(identity["claim"])
print("a, b:", identity["specification"]["definition"]["a"], identity["specification"]["definition"]["b"])